# Исправление ошибок и градиентный бустинг

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Прослеживать последовательное исправление остатков и связывать его с градиентом потери.

## Модели не голосуют независимо

В bagging модели обучаются раздельно. **Бустинг** строит их последовательно: новая пытается исправить ошибки текущей суммы. Запись $F_{t+1}(x)=F_t(x)+\eta h_t(x)$ означает добавление небольшой корректировки, η — темп обучения. Базовая модель hₜ часто является неглубоким деревом.

Для квадратичной ошибки направление исправления — остаток y−F(x). Дерево учат предсказывать эти остатки, затем добавляют часть его ответа. Для других дифференцируемых потерь используют отрицательный градиент по текущему прогнозу. Поэтому метод называется градиентным бустингом; он не обязан буквально предсказывать разности y−F во всех задачах.

Согласуем знак с [метриками регрессии](../../evaluation-preprocessing/regression-metrics/): ошибка e=F−y, остаток r=y−F=−e. Для потери $\ell=\tfrac12(F-y)^2$ производная по F равна F−y, поэтому отрицательный градиент точно равен остатку. Если убрать множитель 1/2, градиент удвоится; минимум останется тем же, а соответствующий размер шага изменится. Поэтому важно сверять нормировку потери, как и при [регуляризации](../regularization/).

## Разобранный пример: один шаг

Ответы [2,4,8], начальная константа — обучающее среднее 14/3. Остатки [−8/3,−2/3,10/3]. Допустим, маленькое дерево предсказало [−2,−2,3]. При η=0.1 новый прогноз равен [14/3−0.2,14/3−0.2,14/3+0.3]. Первые два ответа снизились, третий вырос. Это направленные поправки, а не новый независимый голос.

Малый шаг делает изменения осторожнее, но обычно требует больше деревьев. Большое число этапов способно подстроиться под шум. Число этапов, глубину и η выбирают совместно по внутренней проверке.

В ноутбуке squared_error задаёт квадратичную потерю GradientBoostingRegressor, max_depth ограничивает деревья, n_estimators — число последовательных добавлений. Начальный прогноз библиотеки и последующие шаги можно исследовать через staged_predict: он выдаёт прогноз после каждого этапа.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
X = np.arange(8, dtype=float).reshape(-1, 1)
y = np.array([0., 0., 1., 1., 4., 4., 9., 9.])
model = GradientBoostingRegressor(n_estimators=5, learning_rate=0.1, max_depth=1, random_state=17).fit(X, y)
stages = list(model.staged_predict(X))
assert len(stages) == 5 and stages[-1].shape == y.shape
assert np.allclose(stages[-1], model.predict(X))
print([float(((p-y)**2).mean()) for p in stages])


## Типичные ошибки

Не путайте знак остатка и ошибки F−y. Не считайте много этапов гарантией улучшения validation.

## Задачи

### 1. Базовая

Чем порядок обучения отличается от bagging?

<details>
<summary>Подсказка</summary>

Следующая модель смотрит на текущее решение.

</details>



### 2. Базовая

Каков остаток для ответа 8 и прогноза 5?

<details>
<summary>Подсказка</summary>

Используем y−F.

</details>



### 3. Прикладная

Текущий прогноз 5, поправка 2, η=0.1. Найдите новый.

<details>
<summary>Подсказка</summary>

Добавьте уменьшенную поправку.

</details>



### 4. Прикладная

При квадратичной потере ответ 2, прогноз 7. В какую сторону нужна поправка?

<details>
<summary>Подсказка</summary>

Остаток −5.

</details>



### 5. Олимпиадная

Всегда ли отрицательный градиент равен y−F?

<details>
<summary>Подсказка</summary>

Он зависит от функции потерь.

</details>



### 6. Олимпиадная

Почему нельзя независимо обучить все этапы одновременно на исходных ответах и назвать это тем же бустингом?

<details>
<summary>Подсказка</summary>

Поздний этап зависит от ранних прогнозов.

</details>



## Итоги и самопроверка

Вручную выполните шаг примера. Как изменятся требования к числу этапов при меньшем η?

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

В бустинге этапы зависимы и последовательно исправляют ошибку суммы.

### Задача 2

3.

### Задача 3

5.2.

### Задача 4

В отрицательную: прогноз нужно уменьшить.

### Задача 5

Не всегда. Для потери (y−F)²/2 отрицательная производная по F равна y−F — обычному остатку. Для (y−F)² она равна 2(y−F): направление то же, масштаб другой. При других потерях, например логарифмической, меняется и само выражение. Следовательно, нужно сначала указать функцию потерь, а затем выводить величины, которые аппроксимирует следующий шаг.

### Задача 6

Исчезнет последовательное исправление: такие модели решали бы другую задачу.